Library Dependencies

In [1]:
!pip install pandas torch transformers scikit-learn tqdm


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Imports and Model Initialization

In [2]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import os
from tqdm import tqdm

# Setup device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize bert-base-cased
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')
model = BertModel.from_pretrained('bert-base-cased').to(device)
model.eval() # Set model to evaluation mode

C:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3547.05it/s]
BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

Load and Prepare Data

In [3]:
df = pd.read_csv('fyp.csv')
df['abstract'] = df['abstract'].fillna('')
df['content'] = df['title'] + " " + df['abstract']
print(f"Loaded {len(df)} documents.")

Loaded 5341 documents.


Generate and Save [CLS] EmbeddingsDefine the Embedding Function

In [4]:
def get_cls_embeddings(texts, batch_size=16):
    all_embeddings = []

    # Process in batches to prevent memory errors
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size].tolist()

        # Tokenize
        inputs = tokenizer(batch_texts, padding=True, truncation=True,
                           max_length=512, return_tensors='pt').to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # Extract the [CLS] token (index 0) for the whole batch
        # vector represents the summary of the Title + Abstract
        cls_vectors = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_vectors)

    return np.vstack(all_embeddings)

# Save/Load Logic
file_path = 'cls_embeddings.npy'

if os.path.exists(file_path):
    print("Loading existing embeddings from disk... (Instant)")
    doc_embeddings = np.load(file_path)
else:
    print("Generating [CLS] embeddings... (This may take 10-20 minutes on CPU)")
    doc_embeddings = get_cls_embeddings(df['content'])
    np.save(file_path, doc_embeddings)
    print(f"Embeddings saved as {file_path}")

Generating [CLS] embeddings... (This may take 10-20 minutes on CPU)


100%|██████████| 334/334 [1:06:28<00:00, 11.94s/it]


Embeddings saved as cls_embeddings.npy


Search Function

In [5]:
def semantic_search(query, top_k=5):
    # Convert query to a [CLS] vector
    inputs = tokenizer(query, return_tensors='pt', truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        query_vec = outputs.last_hidden_state[:, 0, :].cpu().numpy()

    # Compare query vector against all saved document vectors
    similarities = cosine_similarity(query_vec, doc_embeddings).flatten()

    # Get the top results
    top_indices = similarities.argsort()[-top_k:][::-1]

    results = df.iloc[top_indices].copy()
    results['similarity_score'] = similarities[top_indices]

    return results[['title', 'similarity_score', 'webpage_url']]

# TEST
query_text = "machine learning applications in Malaysian agriculture"
results = semantic_search(query_text)
results

,title,similarity_score,webpage_url
3141,Application of Green Building Technology in Ma...,0.955319,https://eprints.tarc.edu.my/11533/
3131,Malaysian House Buyers Acceptance for Smart Bu...,0.948857,https://eprints.tarc.edu.my/11539/
5060,Mechanical Design and Fabrication of Hybrid Wh...,0.948750,https://eprints.tarc.edu.my/8956/
4335,Applications of Information Technology (IT) in...,0.947975,https://eprints.tarc.edu.my/11577/
4618,Implementation of TRIZ in Manufacturing Indust...,0.944737,https://eprints.tarc.edu.my/11016/
